<a href="https://colab.research.google.com/github/VisheshKamble/agent-researcher/blob/main/agentresearcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install langgraph langchain langchain-groq langchain-community tavily-python chromadb arxiv python-dotenv requests beautifulsoup4 -q
print("All packages installed ")

All packages installed 


In [18]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

print("GROQ_API_KEY loaded:", os.environ["GROQ_API_KEY"][:8] + "...")
print("TAVILY_API_KEY loaded:", os.environ["TAVILY_API_KEY"][:8] + "...")
print("Keys loaded ")

GROQ_API_KEY loaded: gsk_KlWK...
TAVILY_API_KEY loaded: tvly-dev...
Keys loaded 


In [19]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.3
)

print("Groq LLM ready ")

Groq LLM ready 


In [20]:
response = llm.invoke("Say hello in one sentence.")
print("Groq says:", response.content)

Groq says: Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [21]:
from typing import TypedDict, List

class ResearchState(TypedDict):
    topic: str
    subtasks: List[str]
    search_results: List[dict]
    extracted_content: List[dict]
    verified_facts: List[str]
    final_report: str
    status: str

print("State defined ")

State defined 


In [22]:
def planner_agent(state: ResearchState) -> ResearchState:
    print(" Planner agent running...")

    prompt = f"""
    You are a research planner. Break down this research topic into 4-5 specific subtasks.

    Topic: {state['topic']}

    Return ONLY a Python list of strings like:
    ["subtask 1", "subtask 2", "subtask 3", "subtask 4"]

    Make each subtask specific and searchable.
    """

    response = llm.invoke(prompt)

    import ast, re
    content = response.content
    list_match = re.search(r'\[.*?\]', content, re.DOTALL)
    if list_match:
        subtasks = ast.literal_eval(list_match.group())
    else:
        subtasks = [
            f"latest developments in {state['topic']}",
            f"key players and companies in {state['topic']}",
            f"challenges and limitations of {state['topic']}",
            f"future outlook of {state['topic']}"
        ]

    print(f"  Planner created {len(subtasks)} subtasks:")
    for i, task in enumerate(subtasks, 1):
        print(f"    {i}. {task}")

    return {**state, "subtasks": subtasks, "status": "planned"}

print("Planner agent defined ")

Planner agent defined 


In [24]:
from tavily import TavilyClient

def searcher_agent(state: ResearchState) -> ResearchState:
    print("\n Searcher agent running...")

    client = TavilyClient(api_key=TAVILY_API_KEY)
    all_results = []

    for subtask in state['subtasks']:
        try:
            results = client.search(
                query=subtask,
                max_results=5,
                search_depth="basic"
            )
            for r in results.get('results', []):
                all_results.append({
                    "title": r.get("title", ""),
                    "url": r.get("url", ""),
                    "content": r.get("content", ""),
                    "subtask": subtask
                })
            print(f"  ✓ Found {len(results.get('results', []))} results for: {subtask[:50]}...")
        except Exception as e:
            print(f"  ✗ Search failed for '{subtask}': {e}")

    print(f"\n  Total results collected: {len(all_results)} ")
    return {**state, "search_results": all_results, "status": "searched"}

print("Searcher agent defined ")

Searcher agent defined 


In [25]:
import requests
from bs4 import BeautifulSoup

def reader_agent(state: ResearchState) -> ResearchState:
    print("\n Reader agent running...")

    extracted = []
    seen_urls = set()

    for result in state['search_results'][:8]:
        url = result['url']
        if url in seen_urls:
            continue
        seen_urls.add(url)

        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            resp = requests.get(url, timeout=8, headers=headers)
            soup = BeautifulSoup(resp.text, 'html.parser')

            for tag in soup(["script", "style", "nav", "footer", "header"]):
                tag.decompose()

            raw_text = soup.get_text(separator=' ', strip=True)
            raw_text = ' '.join(raw_text.split())[:3000]

            prompt = f"""
            Extract the most relevant facts about: {state['topic']}

            From this text:
            {raw_text}

            Return 3-5 key factual points as a numbered list.
            Be specific and include data/numbers where available.
            """

            extract = llm.invoke(prompt)

            extracted.append({
                "url": url,
                "title": result['title'],
                "key_points": extract.content,
                "subtask": result['subtask']
            })
            print(f"  ✓ Read: {result['title'][:60]}...")

        except Exception as e:
            print(f"  ✗ Could not read {url[:50]}: {e}")

    print(f"\n  Extracted content from {len(extracted)} sources ")
    return {**state, "extracted_content": extracted, "status": "read"}

print("Reader agent defined ")

Reader agent defined 


In [26]:
def fact_checker_agent(state: ResearchState) -> ResearchState:
    print("\n Fact checker agent running...")

    all_content = ""
    for i, source in enumerate(state['extracted_content'], 1):
        all_content += f"\nSource {i} ({source['url']}):\n{source['key_points']}\n"

    prompt = f"""
    You are a fact checker reviewing research about: {state['topic']}

    Here are extracted points from multiple sources:
    {all_content}

    Your job:
    1. Identify facts confirmed by MULTIPLE sources (mark as ✓ Verified)
    2. Flag facts from only ONE source (mark as ⚠ Unverified)
    3. Remove contradictions and outdated information

    Return a clean, grouped list of facts organized by subtopic.
    """

    response = llm.invoke(prompt)
    verified = [f.strip() for f in response.content.split('\n')
                if f.strip() and len(f.strip()) > 20]

    print(f"  Fact checker processed {len(verified)} fact lines ")
    return {**state, "verified_facts": verified, "status": "verified"}

print("Fact checker agent defined ")

Fact checker agent defined 


In [28]:
def writer_agent(state: ResearchState) -> ResearchState:
    print("\n Writer agent running...")

    facts_text = '\n'.join(state['verified_facts'])
    sources = [s['url'] for s in state['extracted_content']]
    sources_text = '\n'.join([f"{i+1}. {url}" for i, url in enumerate(sources)])

    prompt = f"""
    Write a comprehensive research report on: {state['topic']}

    Use these verified facts:
    {facts_text}

    Structure the report with these sections:

    ## Executive Summary
    (2-3 paragraph overview)

    ## Key Findings
    (most important discoveries with data)

    ## Detailed Analysis
    (deep dive by subtopic)

    ## Current Landscape
    (key players, current state)

    ## Challenges & Limitations
    (existing problems)

    ## Future Outlook
    (where this is heading)

    ## Conclusion
    (wrap up)

    Write professionally. Use specific data and numbers where available.
    """

    response = llm.invoke(prompt)

    full_report = f"""# Research Report: {state['topic']}
---
*Generated by Agent Researcher | {len(state['extracted_content'])} sources analyzed*

{response.content}

---
## Sources
{sources_text}
"""

    print("Report written ")
    return {**state, "final_report": full_report, "status": "complete"}

print("Writer agent defined ")

Writer agent defined 


In [29]:
from langgraph.graph import StateGraph, END

def build_graph():
    graph = StateGraph(ResearchState)

    graph.add_node("planner", planner_agent)
    graph.add_node("searcher", searcher_agent)
    graph.add_node("reader", reader_agent)
    graph.add_node("fact_checker", fact_checker_agent)
    graph.add_node("writer", writer_agent)

    graph.set_entry_point("planner")
    graph.add_edge("planner", "searcher")
    graph.add_edge("searcher", "reader")
    graph.add_edge("reader", "fact_checker")
    graph.add_edge("fact_checker", "writer")
    graph.add_edge("writer", END)

    return graph.compile()

app = build_graph()
print("LangGraph pipeline built ")
print("\nFlow: Planner → Searcher → Reader → Fact Checker → Writer → Done")

LangGraph pipeline built 

Flow: Planner → Searcher → Reader → Fact Checker → Writer → Done


In [30]:
def run_research(topic: str):
    print(f"\n{'='*60}")
    print(f"RESEARCHING: {topic}")
    print(f"{'='*60}\n")

    initial_state = ResearchState(
        topic=topic,
        subtasks=[],
        search_results=[],
        extracted_content=[],
        verified_facts=[],
        final_report="",
        status="starting"
    )

    final_state = app.invoke(initial_state)
    return final_state['final_report']

report = run_research("What is the current state of agentic AI in 2025?")
print("\n" + "="*60)
print("REPORT GENERATED ")
print("="*60)


RESEARCHING: What is the current state of agentic AI in 2025?

 Planner agent running...
  Planner created 5 subtasks:
    1. Identify recent publications on agentic AI from 2023 to 2025
    2. Analyze current applications of agentic AI in industries such as healthcare and finance
    3. Research advancements in agentic AI frameworks and architectures
    4. Examine the role of agentic AI in emerging technologies like robotics and autonomous vehicles
    5. Investigate current challenges and limitations in developing agentic AI systems

 Searcher agent running...
  ✓ Found 5 results for: Identify recent publications on agentic AI from 20...
  ✓ Found 5 results for: Analyze current applications of agentic AI in indu...
  ✓ Found 5 results for: Research advancements in agentic AI frameworks and...
  ✓ Found 5 results for: Examine the role of agentic AI in emerging technol...
  ✓ Found 5 results for: Investigate current challenges and limitations in ...

  Total results collected: 25 

 

In [31]:
from IPython.display import Markdown, display
display(Markdown(report))

# Research Report: What is the current state of agentic AI in 2025?
---
*Generated by Agent Researcher | 8 sources analyzed*

## Executive Summary
The current state of agentic AI in 2025 is characterized by significant growth and adoption across various industries. Agentic AI refers to artificial intelligence systems that can perform tasks autonomously, making decisions and taking actions without human intervention. The adoption rate of agentic AI in enterprise applications is expected to increase substantially, from less than 1% in 2024 to 33% by 2028. This growth is driven by the potential of agentic AI to automate operations, improve decision-making, and enhance customer experience.

The agentic AI market is expected to grow significantly, with the healthcare market valued at USD 538.51 million in 2024 and expected to reach close to USD 5 billion by 2030. Major players have debuted their visions of agentic AI in 2025, marking a new phase in AI development. However, companies will need to address several challenges and limitations, including data silos, informal context, intellectual property, privacy, and security concerns, to ensure safe and productive collaboration between agents.

As the pace of AI innovation is expected to be rapid, fitful, and hard-to-predict, companies will need to maintain an architectural North Star while sustaining progress with fit-for-purpose, domain-specific, and human-in-the-loop builds. With the right approach, agentic AI has the potential to bring significant value to various industries, and its adoption is expected to continue to grow in the coming years.

## Key Findings
The key findings of this research include:
* The adoption rate of agentic AI in enterprise applications is expected to increase from less than 1% in 2024 to 33% by 2028.
* The agentic AI market is expected to grow, with the healthcare market valued at USD 538.51 million in 2024 and expected to reach close to USD 5 billion by 2030.
* Agentic AI is being adopted across various sectors, including financial and healthcare, to automate operations, improve decision-making, and enhance customer experience.
* Companies will need to address data silos, informal context, intellectual property, privacy, and security concerns to ensure safe and productive collaboration between agents.

## Detailed Analysis
### Adoption and Market
The adoption rate of agentic AI in enterprise applications is expected to increase significantly, from less than 1% in 2024 to 33% by 2028. This growth is driven by the potential of agentic AI to automate operations, improve decision-making, and enhance customer experience. The agentic AI market is expected to grow, with the healthcare market valued at USD 538.51 million in 2024 and expected to reach close to USD 5 billion by 2030.

### Characteristics and Definition
Agentic AI refers to artificial intelligence systems that can perform tasks autonomously, making decisions and taking actions without human intervention. Agentic AI systems are designed to operate with minimal human intervention, allowing them to set goals, make decisions, take actions, and adapt based on feedback.

### Industry Applications
Agentic AI is being adopted across various sectors, including financial and healthcare, to automate operations, improve decision-making, and enhance customer experience. In healthcare, agentic AI can automate administrative tasks such as data entry, claims processing, and patient scheduling.

## Current Landscape
The current landscape of agentic AI is characterized by the debut of major players' visions of agentic AI in 2025, marking a new phase in AI development. The pace of AI innovation is expected to be rapid, fitful, and hard-to-predict, with companies needing to maintain an architectural North Star while sustaining progress with fit-for-purpose, domain-specific, and human-in-the-loop builds.

## Challenges & Limitations
Companies will need to address several challenges and limitations to ensure safe and productive collaboration between agents. These challenges include:
* Data silos
* Informal context
* Intellectual property
* Privacy
* Security concerns
* Vendor profit motives

## Future Outlook
The future outlook for agentic AI is promising, with the potential to bring significant value to various industries. As the adoption rate of agentic AI continues to grow, companies will need to stay ahead of the curve by investing in research and development, addressing challenges and limitations, and maintaining an architectural North Star.

## Conclusion
In conclusion, the current state of agentic AI in 2025 is characterized by significant growth and adoption across various industries. With the potential to automate operations, improve decision-making, and enhance customer experience, agentic AI is expected to continue to grow in the coming years. However, companies will need to address several challenges and limitations to ensure safe and productive collaboration between agents. By staying ahead of the curve and investing in research and development, companies can unlock the full potential of agentic AI and bring significant value to their industries.

---
## Sources
1. https://www.oecd.org/content/dam/oecd/en/publications/reports/2026/02/the-agentic-ai-landscape-and-its-conceptual-foundations_a9d4b451/396cf758-en.pdf
2. https://aiagentindex.mit.edu/data/2025-AI-Agent-Index.pdf
3. https://www.bain.com/insights/state-of-the-art-of-agentic-ai-transformation-technology-report-2025/
4. https://www.sciencedirect.com/science/article/abs/pii/S0148296325006228
5. https://medium.com/@brian-curry-research/the-rise-of-agentic-ai-a-technical-deep-dive-into-autonomous-ai-systems-in-2025-c2a9355252dd
6. https://www.softwebsolutions.com/resources/use-cases-of-agentic-ai/
7. https://www.gsdcouncil.org/blogs/exploring-real-world-applications-of-agentic-ai-healthcare-finance-and-smart-automation
8. https://www.xenonstack.com/blog/agentic-ai-healthcare-system


In [32]:
with open("research_report.md", "w") as f:
    f.write(report)
print("Saved as research_report.md ")

from google.colab import files
files.download("research_report.md")

Saved as research_report.md 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>